In [6]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

In [3]:
# Load cleaned dataset
df = pd.read_csv(
    "../../data/processed/paysim_clean.csv"
)

print("Cleaned dataset loaded successfully!")
print("Shape:", df.shape)

print("\nColumns:")
print(df.columns.tolist())

Cleaned dataset loaded successfully!
Shape: (6362620, 10)

Columns:
['step', 'type', 'amount', 'nameOrig', 'oldbalanceOrg', 'newbalanceOrig', 'nameDest', 'oldbalanceDest', 'newbalanceDest', 'isFraud']


In [7]:
# -----------------------------------------
# STEP 5: Transaction Amount Transformation
# -----------------------------------------

# Log-transform transaction amount to normalize the skewed data 
df["log_amount"] = np.log1p(df["amount"])

print("\nLog Amount Feature Created!")

print("\nOriginal Amount:")
print(df["amount"].describe().round(2))

print("\nLog Amount:")
print(df["log_amount"].describe().round(4))


Log Amount Feature Created!

Original Amount:
count     6362620.00
mean       179861.90
std        603858.23
min             0.00
25%         13389.57
50%         74871.94
75%        208721.48
max      92445516.64
Name: amount, dtype: float64

Log Amount:
count    6.362620e+06
mean     1.084090e+01
std      1.814500e+00
min      0.000000e+00
25%      9.502300e+00
50%      1.122350e+01
75%      1.224880e+01
max      1.834210e+01
Name: log_amount, dtype: float64


In [8]:
# -----------------------------------------
# STEP 6: Transaction Time Features
# -----------------------------------------

# Extract hour of day
df["hour"] = df["step"] % 24

# Extract simulation day
df["day"] = df["step"] // 24

print("\nTime Features Created!")

print("\nHour Distribution:")
print(df["hour"].value_counts().sort_index())

print("\nDay Range:")
print("Minimum day:", df["day"].min())
print("Maximum day:", df["day"].max())


Time Features Created!

Hour Distribution:
hour
0      71587
1      27111
2       9018
3       2007
4       1241
5       1641
6       3420
7       8988
8      26915
9     283518
10    425729
11    445992
12    483418
13    468474
14    439653
15    416686
16    441612
17    439941
18    580509
19    647814
20    553728
21    247806
22    194555
23    141257
Name: count, dtype: int64

Day Range:
Minimum day: 0
Maximum day: 30


In [11]:
# -----------------------------------------
# STEP 7: Origin Balance Utilization
# -----------------------------------------

# Calculate how much of the origin balance is being used
df["origin_balance_utilization"] = (
    df["amount"] /
    df["oldbalanceOrg"].replace(0, np.nan)
)

print("\nOrigin Balance Utilization Created!")

print("\nOrigin Balance Utilization Statistics:")
print(
    df["origin_balance_utilization"]
    .describe(
        percentiles=[0.25, 0.50, 0.75, 0.90, 0.95, 0.99]
    )
    .round(4)
)

print(
    "\nMissing utilization values:",
    df["origin_balance_utilization"].isna().sum()
)


Origin Balance Utilization Created!

Origin Balance Utilization Statistics:
count    4.260171e+06
mean     1.220401e+02
std      4.149277e+03
min      0.000000e+00
25%      7.460000e-02
50%      7.408000e-01
75%      6.655800e+00
90%      3.948790e+01
95%      1.765315e+02
99%      1.675171e+03
max      3.925476e+06
Name: origin_balance_utilization, dtype: float64

Missing utilization values: 2102449


In [12]:
# -----------------------------------------
# STEP 8: Zero-Balance Indicators
# -----------------------------------------

df["origin_balance_zero"] = (
    df["oldbalanceOrg"] == 0
).astype(int)

df["destination_balance_zero"] = (
    df["oldbalanceDest"] == 0
).astype(int)

print("\nZero-Balance Indicators Created!")

print("\nOrigin Balance Zero:")
print(df["origin_balance_zero"].value_counts())

print("\nDestination Balance Zero:")
print(df["destination_balance_zero"].value_counts())


Zero-Balance Indicators Created!

Origin Balance Zero:
origin_balance_zero
0    4260171
1    2102449
Name: count, dtype: int64

Destination Balance Zero:
destination_balance_zero
0    3658232
1    2704388
Name: count, dtype: int64


In [13]:
# -----------------------------------------
# STEP 9: Full Origin-Balance Transaction
# -----------------------------------------

df["full_origin_balance_transfer"] = (
    (df["amount"] == df["oldbalanceOrg"]) &
    (df["oldbalanceOrg"] > 0)
).astype(int)

print("\nFull Origin-Balance Feature Created!")

print(
    df["full_origin_balance_transfer"]
    .value_counts()
)


Full Origin-Balance Feature Created!
full_origin_balance_transfer
0    6354602
1       8018
Name: count, dtype: int64


In [14]:
# -----------------------------------------
# STEP 10: Destination Account Activity
# -----------------------------------------

df["destination_transaction_count"] = (
    df.groupby("nameDest")["nameDest"]
      .transform("size")
)

print("\nDestination Transaction Count Created!")

print("\nDestination Transaction Count Statistics:")
print(
    df["destination_transaction_count"]
    .describe()
    .round(2)
)


Destination Transaction Count Created!

Destination Transaction Count Statistics:
count    6362620.00
mean          11.19
std           12.40
min            1.00
25%            1.00
50%            7.00
75%           17.00
max          113.00
Name: destination_transaction_count, dtype: float64


In [15]:
# -----------------------------------------
# STEP 11: Time-Aware Destination Activity
# -----------------------------------------

# Sort transactions chronologically
df = df.sort_values("step").reset_index(drop=True)

# Count previous transactions for each destination
df["destination_previous_transactions"] = (
    df.groupby("nameDest").cumcount()
)

print("\nTime-Aware Destination Activity Created!")

print("\nPrevious Destination Transaction Statistics:")
print(
    df["destination_previous_transactions"]
    .describe()
    .round(2)
)


Time-Aware Destination Activity Created!

Previous Destination Transaction Statistics:
count    6362620.00
mean           5.10
std            7.85
min            0.00
25%            0.00
50%            1.00
75%            7.00
max          112.00
Name: destination_previous_transactions, dtype: float64


In [16]:
# -----------------------------------------
# STEP 12: Time-Aware Origin Activity
# -----------------------------------------

# Count previous transactions made by the origin account
df["origin_previous_transactions"] = (
    df.groupby("nameOrig").cumcount()
)

print("\nTime-Aware Origin Activity Created!")

print("\nPrevious Origin Transaction Statistics:")
print(
    df["origin_previous_transactions"]
    .describe()
    .round(2)
)


Time-Aware Origin Activity Created!

Previous Origin Transaction Statistics:
count    6362620.00
mean           0.00
std            0.04
min            0.00
25%            0.00
50%            0.00
75%            0.00
max            2.00
Name: origin_previous_transactions, dtype: float64


In [17]:
# -----------------------------------------
# STEP 13: Origin Balance Depletion
# -----------------------------------------

df["origin_balance_depleted"] = (
    (df["newbalanceOrig"] == 0) &
    (df["oldbalanceOrg"] > 0)
).astype(int)

print("\nOrigin Balance Depletion Feature Created!")

print("\nOrigin Balance Depletion Distribution:")
print(
    df["origin_balance_depleted"]
    .value_counts()
)



Origin Balance Depletion Feature Created!

Origin Balance Depletion Distribution:
origin_balance_depleted
0    4842039
1    1520581
Name: count, dtype: int64


In [18]:
# -----------------------------------------
# STEP 14: Destination Balance Change
# -----------------------------------------

df["destination_balance_change"] = (
    df["newbalanceDest"] - df["oldbalanceDest"]
)

print("\nDestination Balance Change Feature Created!")

print("\nDestination Balance Change Statistics:")
print(
    df["destination_balance_change"]
    .describe()
    .round(2)
)


Destination Balance Change Feature Created!

Destination Balance Change Statistics:
count    6.362620e+06
mean     1.242947e+05
std      8.129391e+05
min     -1.306083e+07
25%      0.000000e+00
50%      0.000000e+00
75%      1.491054e+05
max      1.056878e+08
Name: destination_balance_change, dtype: float64


In [19]:
# -----------------------------------------
# STEP 15: Origin Balance Change
# -----------------------------------------

df["origin_balance_change"] = (
    df["oldbalanceOrg"] - df["newbalanceOrig"]
)

print("\nOrigin Balance Change Feature Created!")

print("\nOrigin Balance Change Statistics:")
print(
    df["origin_balance_change"]
    .describe()
    .round(2)
)


Origin Balance Change Feature Created!

Origin Balance Change Statistics:
count     6362620.00
mean       -21230.56
std        146643.29
min      -1915267.90
25%             0.00
50%             0.00
75%         10150.44
max      10000000.00
Name: origin_balance_change, dtype: float64


In [20]:
# -----------------------------------------
# STEP 16: Transaction-to-Destination Balance Ratio
# -----------------------------------------

df["amount_to_destination_balance"] = (
    df["amount"] /
    df["oldbalanceDest"].replace(0, np.nan)
)

print("\nTransaction-to-Destination Balance Ratio Created!")

print("\nAmount-to-Destination Balance Statistics:")
print(
    df["amount_to_destination_balance"]
    .describe(
        percentiles=[0.25, 0.50, 0.75, 0.90, 0.95, 0.99]
    )
    .round(4)
)

print(
    "\nMissing ratio values:",
    df["amount_to_destination_balance"].isna().sum()
)


Transaction-to-Destination Balance Ratio Created!

Amount-to-Destination Balance Statistics:
count    3.658232e+06
mean     1.030270e+01
std      3.878965e+03
min      0.000000e+00
25%      6.560000e-02
50%      2.112000e-01
75%      5.670000e-01
90%      1.184600e+00
95%      3.605800e+00
99%      3.061070e+01
max      5.542639e+06
Name: amount_to_destination_balance, dtype: float64

Missing ratio values: 2704388


In [21]:
# -----------------------------------------
# STEP 17: Zero Amount Indicator
# -----------------------------------------

df["zero_amount_transaction"] = (
    df["amount"] == 0
).astype(int)

print("\nZero Amount Transaction Feature Created!")

print(
    df["zero_amount_transaction"].value_counts()
)


Zero Amount Transaction Feature Created!
zero_amount_transaction
0    6362604
1         16
Name: count, dtype: int64


In [22]:
# -----------------------------------------
# STEP 18: Save Feature-Engineered Dataset
# -----------------------------------------

# Remove future-leaking lifetime destination count
df = df.drop(
    columns=["destination_transaction_count"],
    errors="ignore"
)

output_path = "../../data/processed/paysim_features.csv"

df.to_csv(
    output_path,
    index=False
)

print("\nFeature-engineered dataset saved successfully!")
print("File:", output_path)
print("Shape:", df.shape)

print("\nFinal Columns:")
print(df.columns.tolist())


Feature-engineered dataset saved successfully!
File: ../../data/processed/paysim_features.csv
Shape: (6362620, 24)

Final Columns:
['step', 'type', 'amount', 'nameOrig', 'oldbalanceOrg', 'newbalanceOrig', 'nameDest', 'oldbalanceDest', 'newbalanceDest', 'isFraud', 'log_amount', 'hour', 'day', 'origin_balance_utilization', 'origin_balance_zero', 'destination_balance_zero', 'full_origin_balance_transfer', 'destination_previous_transactions', 'origin_previous_transactions', 'origin_balance_depleted', 'destination_balance_change', 'origin_balance_change', 'amount_to_destination_balance', 'zero_amount_transaction']


In [23]:
print(len(df.columns))

24
